# Feature Engineering

In this notebook, we convert cleaned transactional and master data into
model-ready features that capture demand patterns, seasonality,
price effects, promotions, and supplier risk.

Each feature is created with a clear business purpose.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [ ]:
sales_df = pd.read_csv("data/processed/sales_clean.csv")
inventory_df = pd.read_csv("data/processed/inventory_clean.csv")
products_df = pd.read_csv("data/processed/products_clean.csv")
suppliers_df = pd.read_csv("data/processed/suppliers_clean.csv")

sales_df["date"] = pd.to_datetime(sales_df["date"])
inventory_df["snapshot_date"] = pd.to_datetime(inventory_df["snapshot_date"])

## Dataset Preparation

We merge sales, product, and supplier data to create a single
analysis-ready dataset.

In [ ]:
df = (
    sales_df
    .merge(products_df, on="sku_id", how="left")
    .merge(suppliers_df, on="supplier_id", how="left")
)

df.head()

## Calendar-Based Features

These features help capture regular purchasing patterns such as
month-start effects, festive periods, and quarterly cycles.

In [ ]:
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year

# Month start indicator (salary effect proxy)
df["is_month_start"] = df["date"].dt.day <= 5

## Lagged Demand Features

Lag features allow models to learn from recent demand history.
Weekly lags are used since inventory planning is done weekly.

In [ ]:
df = df.sort_values(["sku_id", "date"])

for lag in [1, 2, 4]:
    df[f"lag_{lag}_week_sales"] = (
        df.groupby("sku_id")["units_sold"].shift(lag)
    )

## Rolling Demand Statistics

Rolling statistics capture local demand trends and volatility,
which are critical for safety stock calculations.

In [ ]:
df["rolling_4w_mean"] = (
    df.groupby("sku_id")["units_sold"]
    .rolling(window=4)
    .mean()
    .reset_index(level=0, drop=True)
)

df["rolling_4w_std"] = (
    df.groupby("sku_id")["units_sold"]
    .rolling(window=4)
    .std()
    .reset_index(level=0, drop=True)
)

## Price & Promotion Features

Price changes and discounts strongly influence demand,
especially during campaigns and festive periods.

In [ ]:
# Price gap (absolute discount)
df["price_gap"] = df["mrp"] - df["selling_price"]

# Discount percentage
df["discount_pct"] = df["price_gap"] / df["mrp"]

# Promotion flag (assuming discount > 0 implies promotion)
df["is_promo"] = df["discount_pct"] > 0

## Seasonality Indicators

Some categories experience strong seasonal demand.
We encode simple seasonality signals.

In [ ]:
# Example seasonal mapping (adjust if needed)
winter_months = [11, 12, 1]
summer_months = [4, 5, 6]

df["is_winter_season"] = df["month"].isin(winter_months)
df["is_summer_season"] = df["month"].isin(summer_months)

## Supplier Lead Time & Risk Features

Lead time variability directly impacts reorder point and safety stock.

In [ ]:
# Lead time is already present; create variability proxy
df["lead_time_days"] = df["lead_time_days"]

# Supplier risk bucket
df["lead_time_risk"] = pd.cut(
    df["lead_time_days"],
    bins=[0, 5, 10, 20, np.inf],
    labels=["low", "medium", "high", "very_high"]
)

## Demand Volatility Feature

Volatility is measured using the coefficient of variation (CV).
High CV indicates unpredictable demand.

In [ ]:
df["demand_cv"] = df["rolling_4w_std"] / df["rolling_4w_mean"]

## Final Feature Review

Lag and rolling features naturally introduce missing values.
These rows are removed for modeling purposes.

In [ ]:
feature_df = df.dropna().reset_index(drop=True)

feature_df.shape

## Save Feature-Engineered Dataset

This dataset will be used for SKU segmentation and forecasting.

In [ ]:
feature_df.to_csv("data/processed/feature_engineered_data.csv", index=False)